# Motor de Web Scraping — Ofertas Financieras Competencia Automoción
**Honda Financial Services | TFM BSM Barcelona**

Extrae ofertas de financiación (TIN, TAE, comisión de apertura, cuota, plazo, etc.) de webs de competidores usando scraping + LLM (Claude).

## 1. Instalación de dependencias (ejecutar solo en Colab)

In [ ]:
# Ejecuta esta celda la primera vez en Google Colab (tarda ~1 minuto)
!pip install -q requests beautifulsoup4 selenium pandas webdriver-manager openai
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y -q ./google-chrome-stable_current_amd64.deb
print("Dependencias instaladas correctamente")

## 2. Imports y configuración

In [ ]:
import requests
import time
import json
import re
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from openai import OpenAI

print("Librerías cargadas correctamente")

In [ ]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

CAMPOS_OFERTA = [
    "marca",
    "modelo",
    "precio_vehiculo",
    "cuota_mensual",
    "plazo_meses",
    "entrada",
    "tin",
    "tae",
    "comision_apertura",
    "valor_residual",
    "importe_financiado",
    "tipo_financiacion",
    "fecha_fin_oferta",
    "url",
    "fecha_extraccion"
]

print(f"API Key OpenAI cargada: {'OK' if OPENAI_API_KEY else 'ERROR — revisa los Secrets'}")

## 3. Funciones de scraping

In [ ]:
def scrape_estatico(url, reintentos=3, pausa=2):
    for intento in range(reintentos):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            print(f"  [intento {intento+1}/{reintentos}] Error en {url}: {e}")
            time.sleep(pausa * (intento + 1))
    return None


def crear_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")
    driver_path = ChromeDriverManager().install()
    return webdriver.Chrome(service=Service(driver_path), options=options)


def scroll_hasta_el_final(driver, pausas=8):
    for i in range(pausas):
        driver.execute_script("window.scrollBy(0, document.body.scrollHeight);")
        time.sleep(1.5)
    driver.execute_script("window.scrollTo(0, 0);")


def scrape_dinamico(url, espera_extra=3, scroll=False):
    driver = crear_driver()
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(espera_extra)
        if scroll:
            scroll_hasta_el_final(driver)
            time.sleep(2)
        return driver.page_source
    except Exception as e:
        print(f"  Error Selenium en {url}: {e}")
        return None
    finally:
        driver.quit()


KEYWORDS_LEGALES = ["tin", "tae", "comisión de apertura", "importe financiado",
                    "importe total", "tipo de interés", "coste total", "cuota final",
                    "valor residual", "financiado por", "cuotas de"]

def html_a_texto(html, seccion_especial=None, max_chars_legal=6000):
    """
    Extrae del HTML solo los fragmentos relevantes:
    - seccion_especial: busca esa cadena y extrae desde ahí (ej. Renault)
    - Si no, detecta el bloque más denso en keywords legales
    - Combina cabecera (modelo+cuota) + bloque legal
    """
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "noscript"]):
        tag.decompose()
    texto = soup.get_text(separator=" ", strip=True)
    texto = re.sub(r'\s+', ' ', texto)

    cabecera = texto[:500]

    # Caso especial: extraer desde una sección concreta (ej. Renault)
    if seccion_especial:
        idx = texto.find(seccion_especial)
        if idx != -1:
            bloque = texto[idx:idx + max_chars_legal]
            print(f"  Sección '{seccion_especial[:30]}' encontrada en pos {idx}")
            return cabecera + " [...] " + bloque
        else:
            print(f"  AVISO: Sección especial no encontrada, usando detección automática")

    # Detección automática del bloque legal más denso
    texto_lower = texto.lower()
    mejor_pos, mejor_score = -1, 0
    ventana = 500
    for i in range(0, len(texto) - ventana, 200):
        score = sum(texto_lower[i:i+ventana].count(kw) for kw in KEYWORDS_LEGALES)
        if score > mejor_score:
            mejor_score = score
            mejor_pos = i

    if mejor_pos > 0 and mejor_score > 0:
        inicio = max(0, mejor_pos - 100)
        bloque = texto[inicio:min(len(texto), inicio + max_chars_legal)]
    else:
        bloque = texto[-max_chars_legal:]  # fallback: final de la página

    resultado = cabecera + " [...] " + bloque
    print(f"  Texto enviado al LLM: {len(resultado)} chars (densidad legal: {mejor_score})")
    return resultado


print("Funciones de scraping definidas")

## 4. Extracción con LLM (Claude)

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

PROMPT_SISTEMA = """Eres un experto en análisis de ofertas de financiación de automóviles en España.
Tu tarea es extraer información estructurada de textos de páginas web de concesionarios.
Devuelve SIEMPRE un JSON válido con los campos indicados.
Si un campo no aparece en el texto, devuelve null para ese campo.
No inventes datos. Solo extrae lo que esté explícitamente en el texto."""


def extraer_oferta_con_llm(texto, marca, url):
    prompt = f"""Analiza el siguiente texto de la web de {marca} ({url}) y extrae TODAS las ofertas de financiación.

Para cada oferta devuelve un JSON con estos campos:
- modelo: nombre del modelo
- tipo_combustible: "gasolina", "diésel", "híbrido", "híbrido enchufable", "eléctrico" o null
- precio_vehiculo: precio total (número)
- cuota_mensual: cuota mensual en € (número)
- plazo_meses: duración en meses (número)
- entrada: entrada inicial en € (número, 0 si no hay)
- tin: TIN en % (número)
- tae: TAE en % (número)
- comision_apertura: importe en € (número, 0 si es gratuita)
- porcentaje_comision_apertura: comisión de apertura en % sobre el capital (número o null)
- valor_residual: valor residual o cuota final en € (número o null)
- importe_financiado: capital financiado en € (número)
- tipo_financiacion: "credito", "leasing", "renting", "PCP" u otro
- fecha_fin_oferta: fecha límite en YYYY-MM-DD (string o null)

Devuelve SOLO este JSON sin texto adicional:
{{"ofertas": [ {{...}} ]}}

TEXTO:
{texto}"""

    try:
        respuesta = client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens=3000,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": PROMPT_SISTEMA},
                {"role": "user", "content": prompt}
            ]
        )
        datos = json.loads(respuesta.choices[0].message.content)
        ofertas = datos.get("ofertas", [])
        for oferta in ofertas:
            oferta["marca"] = marca
            oferta["url"] = url
            oferta["fecha_extraccion"] = str(date.today())
        return ofertas
    except Exception as e:
        print(f"  Error LLM para {url}: {e}")
        return []


print("Cliente OpenAI (gpt-4o-mini) configurado")

## 5. Pipeline completo: scraping + extracción LLM

In [ ]:
def procesar_url(url, marca, usar_selenium=False, scroll=False, seccion_especial=None):
    print(f"Procesando: {marca} — {url}")
    html = scrape_dinamico(url, scroll=scroll) if usar_selenium else scrape_estatico(url)
    if not html:
        print(f"  No se pudo descargar {url}")
        return []
    texto = html_a_texto(html, seccion_especial=seccion_especial)
    ofertas = extraer_oferta_con_llm(texto, marca, url)
    print(f"  Ofertas encontradas: {len(ofertas)}")
    return ofertas


print("Pipeline definido")

## 6. URLs de la competencia

In [ ]:
COMPETENCIA = {
    "TOYOTA": {
        "selenium": False, "scroll": False, "seccion_especial": None,
        "urls": [
            "https://www.toyota.es/promociones/toyota-c-hr-plus-easy-plus",
            "https://www.toyota.es/promociones/toyota-c-hr-140h-advance-easy-plus",
            "https://www.toyota.es/promociones/toyota-c-hr-plug-in-hybrid-220ph-advance-easy-plus",
            "https://www.toyota.es/promociones/yaris-ng-active-tech-easy-plus",
            "https://www.toyota.es/promociones/yaris-cross-hybrid-style-easy-plus",
            "https://www.toyota.es/promociones/corolla-hybrid-140h-active-plus-easy-plus",
            "https://www.toyota.es/promociones/corolla-sedan-hybrid-140h-style-plus-easy-plus",
            "https://www.toyota.es/promociones/corolla-touring-sports-hybrid-140h-easy-plus",
            "https://www.toyota.es/promociones/corolla-cross-hybrid-style-easy-plus",
            "https://www.toyota.es/promociones/toyota-bz4x-electric-4x2-advance-easy-plus",
            "https://www.toyota.es/promociones/aygo-x-cross-play-easy",
            "https://www.toyota.es/promociones/rav4-hybrid-220h-2x4-advance-easy-plus",
            "https://www.toyota.es/promociones/rav4-plug-in-hybrid-300ph-advance-easy-plus"
        ]
    },
    "VOLKSWAGEN": {
        "selenium": True, "scroll": True, "seccion_especial": None,
        "urls": ["https://www.volkswagen.es/es/ofertas.html"]
    },
    "PEUGEOT": {
        "selenium": True, "scroll": True, "seccion_especial": None,
        "urls": ["https://www.peugeot.es/comprar/ofertas-del-momento.html"]
    },
    "RENAULT": {
        "selenium": False, "scroll": False,
        "seccion_especial": "CONDICIONES LEGALES PARA PENÍNSULA Y BALEARES",
        "urls": [
            "https://promociones.renault.es/particulares/clio/",
            "https://promociones.renault.es/particulares/captur/",
            "https://promociones.renault.es/particulares/symbioz/",
            "https://promociones.renault.es/particulares/symbioz-glp/",
            "https://promociones.renault.es/particulares/austral/",
            "https://promociones.renault.es/particulares/arkana/",
            "https://promociones.renault.es/particulares/espace/",
            "https://promociones.renault.es/particulares/rafale/",
            "https://promociones.renault.es/particulares/rafale-phev/"
        ]
    },
    "NISSAN": {
        "selenium": True, "scroll": True, "seccion_especial": None,
        "urls": ["https://www.nissan.es/vehiculos/ofertas.html"]
    },
    "HYUNDAI": {
        "selenium": True, "scroll": True, "seccion_especial": None,
        "urls": [
            "https://www.hyundai.com/es/es/modelos/kona.html",
            "https://www.hyundai.com/es/es/modelos/tucson.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-bayon.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-i10.html",
            "https://www.hyundai.com/es/es/modelos/i20.html",
            "https://www.hyundai.com/es/es/modelos/i30.html",
            "https://www.hyundai.com/es/es/modelos/i30-fastback.html",
            "https://www.hyundai.com/es/es/modelos/i30wagon.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-santafe-hev.html",
            "https://www.hyundai.com/es/es/modelos/kona-hibrido.html",
            "https://www.hyundai.com/es/es/modelos/tucson-hibrido.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-santafe-phev.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-tucson-phev.html",
            "https://www.hyundai.com/es/es/modelos/inster.html",
            "https://www.hyundai.com/es/es/modelos/kona-electrico.html",
            "https://www.hyundai.com/es/es/modelos/ioniq5.html",
            "https://www.hyundai.com/es/es/modelos/ioniq6.html",
            "https://www.hyundai.com/es/es/modelos/ioniq9.html"
        ]
    },
    "AUDI": {
        "selenium": True, "scroll": True, "seccion_especial": None,
        "urls": ["https://www.audi.es/es/compra/promociones/"]
    }
}

print(f"Configuradas {sum(len(v['urls']) for v in COMPETENCIA.values())} URLs de {len(COMPETENCIA)} marcas")

## 7. Ejecución del scraping

In [ ]:
# Ejecutar scraping completo de todas las marcas
# Para probar con una sola marca: MARCAS_A_EJECUTAR = ["TOYOTA"]
MARCAS_A_EJECUTAR = list(COMPETENCIA.keys())

todas_las_ofertas = []

for marca in MARCAS_A_EJECUTAR:
    config = COMPETENCIA[marca]
    print(f"\n{'='*50}")
    print(f"MARCA: {marca}")
    print(f"{'='*50}")

    for url in config["urls"]:
        ofertas = procesar_url(url, marca, usar_selenium=config["selenium"])
        todas_las_ofertas.extend(ofertas)
        time.sleep(2)  # Pausa educada entre peticiones

print(f"\nTotal de ofertas extraídas: {len(todas_las_ofertas)}")

MARCAS_A_EJECUTAR = list(COMPETENCIA.keys())
# Para probar una sola marca: MARCAS_A_EJECUTAR = ["NISSAN"]

todas_las_ofertas = []

for marca in MARCAS_A_EJECUTAR:
    config = COMPETENCIA[marca]
    print(f"\n{'='*50}\nMARCA: {marca}\n{'='*50}")
    for url in config["urls"]:
        ofertas = procesar_url(
            url, marca,
            usar_selenium=config["selenium"],
            scroll=config.get("scroll", False),
            seccion_especial=config.get("seccion_especial")
        )
        todas_las_ofertas.extend(ofertas)
        time.sleep(2)

print(f"\n{'='*50}")
print(f"RESUMEN: {len(todas_las_ofertas)} ofertas brutas extraídas")
marcas_con_datos = set(o["marca"] for o in todas_las_ofertas)
print(f"Marcas CON datos: {marcas_con_datos}")
marcas_sin_datos = set(MARCAS_A_EJECUTAR) - marcas_con_datos
if marcas_sin_datos:
    print(f"Marcas SIN datos: {marcas_sin_datos}")

In [ ]:
df_bruto = pd.DataFrame(todas_las_ofertas)

if df_bruto.empty:
    print("No se han extraído ofertas.")
else:
    # Filtro: por cada (marca, modelo) quedarse con la oferta que tenga TIN
    # Si hay varias con TIN, quedarse con la primera
    df_bruto["tiene_tin"] = df_bruto["tin"].notna()
    df_filtrado = (
        df_bruto
        .sort_values("tiene_tin", ascending=False)
        .drop_duplicates(subset=["marca", "modelo"], keep="first")
        .drop(columns=["tiene_tin"])
        .reset_index(drop=True)
    )

    # Ordenar columnas
    CAMPOS_ORDENADOS = [
        "marca", "modelo", "tipo_combustible", "precio_vehiculo",
        "cuota_mensual", "plazo_meses", "entrada", "tin", "tae",
        "comision_apertura", "porcentaje_comision_apertura",
        "valor_residual", "importe_financiado", "tipo_financiacion",
        "fecha_fin_oferta", "url", "fecha_extraccion"
    ]
    cols = [c for c in CAMPOS_ORDENADOS if c in df_filtrado.columns]
    df = df_filtrado[cols]

    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_rows", 100)

    print(f"Ofertas brutas: {len(df_bruto)} → tras filtro (1 por modelo con TIN): {len(df)}")
    print(f"\nModelos por marca:")
    print(df.groupby("marca")["modelo"].count().to_string())
    display(df)

In [ ]:
# Exportar a CSV
nombre_archivo = f"ofertas_competencia_{date.today().strftime('%Y%m%d')}.csv"
df.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
print(f"Guardado en: {nombre_archivo}")

# En Colab, descargar el archivo:
# from google.colab import files
# files.download(nombre_archivo)

In [ ]:
# Resumen comparativo por marca
if not df.empty and "marca" in df.columns:
    resumen = df.groupby("marca").agg(
        num_ofertas=("modelo", "count"),
        tin_medio=("tin", "mean"),
        tae_medio=("tae", "mean"),
        cuota_min=("cuota_mensual", "min"),
        cuota_max=("cuota_mensual", "max")
    ).round(2)
    print(resumen)